# 04 — Train: LightGBM regression

Fits one fixed multi-output LightGBM model (via `MultiOutputRegressor`) for the configured target station and evaluates it once on the test feature artifact.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview, model configuration, and test metrics

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins every knob the notebook uses. The boosting configuration is one hand-picked guess, not the outcome of a search.

**What the imports provide**

- `LGBMRegressor` — LightGBM's gradient-boosting estimator. Unlike XGBoost it grows trees *leaf-wise* (always splitting the leaf with the highest expected gain) rather than level-by-level, which reaches a given accuracy with fewer, deeper-but-narrower trees.
- `MultiOutputRegressor` — a scikit-learn wrapper that clones an estimator once per target column. It is required because `LGBMRegressor` accepts only a 1-D target.
- `mean_absolute_error`, `root_mean_squared_error` — the two reported error metrics.
- `feature_column_names()`, `target_column_names()` — the Stage-3 column contract.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 feature Parquets are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | How many scored test rows the final preview table shows. Display only; it has no effect on any metric. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract, read from `feature_column_names()` instead of being hardcoded so the notebook fails loudly if Stage 3 ever changes it. It contains: the raw `water_level`, `imputed`, `precipitation` and `temperature_2m`; 8 water-level lags (1, 3, 6, 12, 24, 48, 72, 168 h); 5 water-level differences (1–24 h); 16 rolling water-level statistics (mean/std/min/max x 6/24/72/168 h); 4 rolling imputation counts; 4 rolling precipitation sums; 4 rolling temperature means plus the 24 h temperature min and max; and 6 calendar Fourier terms. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | The 24 future water levels, one per lead hour. The model emits all of them from a single feature vector — a *direct* multi-horizon setup, with no recursive feeding of its own predictions. |
| `LGBM_N_ESTIMATORS` | `300` | Boosting rounds *per horizon*. With the wrapper below this means 300 trees x 24 models = 7,200 trees in total. |
| `LGBM_MAX_DEPTH` | `6` | Hard cap on tree depth. In LightGBM this is a safety limit rather than the main capacity knob, because growth is leaf-wise: `num_leaves` is what usually binds first. |
| `LGBM_LEARNING_RATE` | `0.05` | Shrinkage on each tree's contribution. Same trade-off as XGBoost: lower is more cautious and needs more rounds. |
| `LGBM_NUM_LEAVES` | `31` | The primary capacity knob. It caps how many leaves a single leaf-wise-grown tree may have. The rule of thumb is to keep it below `2 ** max_depth`; `31 < 2 ** 6 = 64`, so the leaf budget binds before the depth cap does, which is the intended behaviour. |
| `LGBM_SUBSAMPLE` | `0.8` | Fraction of rows sampled per bagging round. On its own this parameter does nothing — LightGBM ignores it unless `subsample_freq` is greater than zero. |
| `LGBM_SUBSAMPLE_FREQ` | `1` | Re-draw the row sample every iteration. This is the switch that actually activates `subsample`; leaving it at its default of `0` would silently disable row bagging. |
| `LGBM_COLSAMPLE_BYTREE` | `0.8` | Fraction of the 53 predictors offered to each tree, de-correlating successive trees. |
| `LGBM_OBJECTIVE` | `regression` | LightGBM's name for the L2 (squared-error) objective, so the trees estimate the conditional mean. |
| `LGBM_RANDOM_STATE` | `42` | Seeds row and feature sampling for reproducibility. |
| `LGBM_N_JOBS` | `-1` | Use all CPU cores. Speed only. |

The fit cell also passes `verbose=-1`, which suppresses LightGBM's per-iteration console output — worth having when the wrapper is about to fit 24 models in a row.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

from src.config import TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
LGBM_N_ESTIMATORS = 300
LGBM_MAX_DEPTH = 6
LGBM_LEARNING_RATE = 0.05
LGBM_NUM_LEAVES = 31
LGBM_SUBSAMPLE = 0.8
LGBM_SUBSAMPLE_FREQ = 1
LGBM_COLSAMPLE_BYTREE = 0.8
LGBM_OBJECTIVE = "regression"
LGBM_RANDOM_STATE = 42
LGBM_N_JOBS = -1
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())

## Shared evaluation cohort

Every stage-4 candidate is fit and scored on exactly the same rows, which is what makes their reported numbers comparable to each other and to the persistence baseline. One row is one *issue time* `t`, and it qualifies only when both of these hold:

1. **Stage 3 marked it `target_valid`.** All 24 future water levels `t+1 … t+24` were actually observed, none of them synthesised. This drops issue times sitting near a data gap, plus the final 24 hours of each artifact, which have no complete future.
2. **All 53 predictors are present.** The lag and rolling features need a complete 168-hour lookback, so the first week of each artifact is a warm-up that can never qualify.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it.

## Helper functions

Three small helpers used by the load / fit / evaluate cells below. All of them take keyword-only `station_id` and `artifact_name` arguments that exist purely so a raised error names the split it came from instead of leaving you to guess.

**`eligible_rows(frame, *, station_id, artifact_name) -> pd.Series`**

- `frame` — one loaded feature artifact (train or test).
- `station_id` — the station the artifact is supposed to describe; used in error messages.
- `artifact_name` — `"train"` or `"test"`, likewise for error messages.

Returns a boolean mask marking the cohort rows defined above. It raises instead of returning a mask when the artifact is missing a contract column, or when a `target_valid` row still carries a null target — that combination is a Stage-3 bug, and silently averaging around it would produce a metric that looks fine and is not.

**`metric_tables(actual, predictions, *, station_id) -> (aggregate, per_horizon)`**

- `actual` — the cohort's `TARGET_COLUMNS` frame, shape `(n_issue_times, 24)`.
- `predictions` — the model's output, in the same shape and the same row order.

Returns two frames. The **aggregate** one pools all `n x 24` values into a single MAE and RMSE. The **per-horizon** one repeats that calculation separately for each lead time, which is what reveals how fast accuracy decays from `t+1` to `t+24`. Both metrics are in the water level's native unit: MAE is the average absolute miss, while RMSE squares the errors before averaging, so it is always at least as large as MAE and is dominated by the worst forecasts — a wide gap between the two means a few large misses rather than uniformly poor accuracy.

**`prediction_preview(frame, predictions) -> pd.DataFrame`**

- `frame` — the scored cohort rows, supplying `timestamp` and the actual targets.
- `predictions` — the matching predicted array.

Returns one frame with each `prediction_target_t_plus_XX` column beside its actual counterpart, aligned on the cohort's original index so no row silently shifts.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon

In [ ]:
def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load feature artifacts

Resolves `data/processed/<station>_train_features.parquet` and `<station>_test_features.parquet` for the station configured in `src.config.TARGET_STATION_ID`, then raises `FileNotFoundError` if either is absent. A missing artifact means stages 1–3 have not been run for this station, and failing here costs a second rather than failing halfway through a fit.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)

## Apply the eligibility cohort

Builds the train and test masks with `eligible_rows()` and keeps only the rows that pass. If either split ends up with no eligible row the notebook stops here, rather than fitting on an empty frame and reporting a metric computed from nothing.

In [ ]:
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

train_rows = train_features.loc[train_mask]
test_rows = test_features.loc[test_mask]

## Fit the fixed multi-output LightGBM model

`LGBMRegressor` accepts only a single target column, so `MultiOutputRegressor` clones the configuration and fits one independent model per horizon — 24 fits in total, each seeing the same 53 predictors but a different target column. `predict` runs all 24 and stacks the results back into one `(n_test, 24)` array in `TARGET_COLUMNS` order.

The trade-off is explicit: every horizon gets its own trees and its own splits, at 24 times the cost of a single fit and with no sharing of structure between neighbouring lead times.

Raw numeric feature values go straight to the trees — no scaling, no imputation (the cohort filter already removed incomplete rows), no tuning, no validation split, no early stopping, no second model.

In [ ]:
lightgbm_model = MultiOutputRegressor(
    LGBMRegressor(
        n_estimators=LGBM_N_ESTIMATORS,
        max_depth=LGBM_MAX_DEPTH,
        learning_rate=LGBM_LEARNING_RATE,
        num_leaves=LGBM_NUM_LEAVES,
        subsample=LGBM_SUBSAMPLE,
        subsample_freq=LGBM_SUBSAMPLE_FREQ,
        colsample_bytree=LGBM_COLSAMPLE_BYTREE,
        objective=LGBM_OBJECTIVE,
        random_state=LGBM_RANDOM_STATE,
        n_jobs=LGBM_N_JOBS,
        verbose=-1,
    )
)
lightgbm_model.fit(train_rows[FEATURE_COLUMNS], train_rows[TARGET_COLUMNS])
test_predictions = lightgbm_model.predict(test_rows[FEATURE_COLUMNS])

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort: aggregate MAE/RMSE, the same two metrics per lead time, and a short preview so the predictions can be eyeballed against their actual targets. There is no second pass and no refitting — what is printed here is the notebook's one and only result.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
model_configuration = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "model": "LGBMRegressor",
            "n_estimators": LGBM_N_ESTIMATORS,
            "max_depth": LGBM_MAX_DEPTH,
            "learning_rate": LGBM_LEARNING_RATE,
            "num_leaves": LGBM_NUM_LEAVES,
            "subsample": LGBM_SUBSAMPLE,
            "subsample_freq": LGBM_SUBSAMPLE_FREQ,
            "colsample_bytree": LGBM_COLSAMPLE_BYTREE,
            "objective": LGBM_OBJECTIVE,
            "random_state": LGBM_RANDOM_STATE,
            "n_jobs": LGBM_N_JOBS,
        }
    ]
)
print(f"LightGBM test results for {station_id}")
display(model_configuration)
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))